In [1]:
import time
import torch
import sys
import types
import functools
fake_qwen3 = types.ModuleType("transformers.models.qwen3")
sys.modules["transformers.models.qwen3"] = fake_qwen3
sys.modules["transformers.models.qwen3.modeling_qwen3"] = fake_qwen3

# 2. Add dummy classes so the from/import doesn't fail
setattr(fake_qwen3, "Qwen3DecoderLayer", object)
setattr(fake_qwen3, "Qwen3ForCausalLM", object)
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer
import pandas as pd
from openai import OpenAI
from tabulate import tabulate
import transformers.utils
import transformers.activations


# Fix for NameError and cached_property ImportError
transformers.utils.cached_property = functools.cached_property

# Fix for PytorchGELUTanh ImportError
if not hasattr(transformers.activations, 'PytorchGELUTanh'):
    transformers.activations.PytorchGELUTanh = transformers.activations.GELUActivation
# --- CONFIGURATION ---
MODEL_PATH = "C:/Users/Ansh/Models/sarvam-1" # Update to your local path
OLLAMA_URL = "http://localhost:11434/v1"
#TRT_URL = "http://localhost:8000/v1"

# The Benchmark Prompt: Logic + Language + Formatting
BENCHMARK_PROMPT = """
Explain the concept of 'Backpropagation' in Deep Learning using a simple analogy in Hindi. 
Then, list 3 reasons why it is essential for training Large Language Models.
"""

def benchmark_api(name, url, model_name, warm_up=3):
    client = OpenAI(base_url=url, api_key="not-needed")
    print(f"\n>>> Testing {name}...")
    
    # Warm-up (Handle Cold Start)
    for _ in range(warm_up):
        client.chat.completions.create(model=model_name, messages=[{"role": "user", "content": "hi"}], max_tokens=1)
    
    start = time.perf_counter()
    ttft, tokens, full_text = None, 0, ""
    
    stream = client.chat.completions.create(
        model=model_name, 
        messages=[{"role": "user", "content": BENCHMARK_PROMPT}], 
        stream=True
    )
    
    for chunk in stream:
        if ttft is None:
            ttft = (time.perf_counter() - start) * 1000
        content = chunk.choices[0].delta.content
        if content:
            tokens += len(content.split()) # Rough token count
            full_text += content
            
    duration = time.perf_counter() - start
    return [name, round(ttft, 2), round(tokens/duration, 2), round(duration, 2)]

def benchmark_native_pytorch(warm_up=3):
    print("\n>>> Testing Native PyTorch (AutoAWQ)...")
    # Load Model
    model = AutoAWQForCausalLM.from_quantized(
    MODEL_PATH, 
    fuse_layers=True, 
    device_map="cuda:0", # Force it directly to the GPU
    offload_folder="offload", # Safety for 4GB cards
    trust_remote_code=True
)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    
    # Warm-up
    inputs = tokenizer("hi", return_tensors="pt").to("cuda")
    for _ in range(warm_up):
        model.generate(**inputs, max_new_tokens=1)
    
    # Real Run
    inputs = tokenizer(BENCHMARK_PROMPT, return_tensors="pt").to("cuda")
    start = time.perf_counter()
    
    # Simple TTFT estimation for native (non-streaming for simplicity)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    
    duration = time.perf_counter() - start
    total_tokens = len(outputs[0]) - len(inputs.input_ids[0])
    
    # Clean up VRAM immediately after test
    del model
    torch.cuda.empty_cache()
    
    return ["Native PyTorch", "N/A (Local)", round(total_tokens/duration, 2), round(duration, 2)]

# --- EXECUTION ---
results = []

# 1. Test TRT-LLM (API)
# try:
#     results.append(benchmark_api("TensorRT-LLM", TRT_URL, "sarvam-1"))
# except Exception as e:
#     print(f"TRT-LLM Server not found: {e}")

# 2. Test Ollama (API)
try:
    results.append(benchmark_api("Ollama", OLLAMA_URL, "mashriram/sarvam-1"))
except Exception as e:
    print(f"Ollama Server not found: {e}")

# 3. Test Native PyTorch
try:
    results.append(benchmark_native_pytorch())
except Exception as e:
    print(f"Native PyTorch failed: {e}")

# --- OUTPUT ---
print("\n" + "="*60)
print("FINAL PERFORMANCE COMPARISON: RTX 3050 (4GB VRAM)")
print("="*60)
headers = ["Runtime", "Warm TTFT (ms)", "Tokens/Sec", "Total Time (s)"]
print(tabulate(results, headers=headers, tablefmt="grid"))

C:\Users\Ansh\AppData\Roaming\Python\Python312\site-packages\awq\__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)



>>> Testing Ollama...

>>> Testing Native PyTorch (AutoAWQ)...


Replacing layers...: 100%|██████████| 28/28 [00:10<00:00,  2.68it/s]

Native PyTorch failed: If passing a string for `device_map`, please choose 'auto', 'balanced', 'balanced_low_0' or 'sequential'.

FINAL PERFORMANCE COMPARISON: RTX 3050 (4GB VRAM)
+-----------+------------------+--------------+------------------+
| Runtime   |   Warm TTFT (ms) |   Tokens/Sec |   Total Time (s) |
+===========+==================+==============+==================+
| Ollama    |           316.28 |        52.04 |             7.15 |
+-----------+------------------+--------------+------------------+


In [1]:
import time
import torch
import sys
import types
import functools
import pandas as pd
from openai import OpenAI
from tabulate import tabulate
import transformers.utils
import transformers.activations

# --- MONKEY PATCHES ---
fake_qwen3 = types.ModuleType("transformers.models.qwen3")
sys.modules["transformers.models.qwen3"] = fake_qwen3
sys.modules["transformers.models.qwen3.modeling_qwen3"] = fake_qwen3
setattr(fake_qwen3, "Qwen3DecoderLayer", object)
setattr(fake_qwen3, "Qwen3ForCausalLM", object)

transformers.utils.cached_property = functools.cached_property
if not hasattr(transformers.activations, 'PytorchGELUTanh'):
    transformers.activations.PytorchGELUTanh = transformers.activations.GELUActivation

from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

# --- CONFIGURATION ---
MODEL_PATH = "C:/Users/Ansh/Models/sarvam-1" 
OLLAMA_URL = "http://localhost:11434/v1"

BENCHMARK_PROMPT = """
Explain the concept of 'Backpropagation' in Deep Learning using a simple analogy in Hindi. 
Then, list 3 reasons why it is essential for training Large Language Models.
"""

def benchmark_api(name, url, model_name, warm_up=3):
    client = OpenAI(base_url=url, api_key="not-needed")
    print(f"\n>>> Testing {name}...")
    
    for _ in range(warm_up):
        client.chat.completions.create(model=model_name, messages=[{"role": "user", "content": "hi"}], max_tokens=1)
    
    start = time.perf_counter()
    ttft, tokens, full_text = None, 0, ""
    
    stream = client.chat.completions.create(
        model=model_name, 
        messages=[{"role": "user", "content": BENCHMARK_PROMPT}], 
        stream=True
    )
    
    for chunk in stream:
        if ttft is None:
            ttft = (time.perf_counter() - start) * 1000
        content = chunk.choices[0].delta.content
        if content:
            tokens += len(content.split()) 
            full_text += content
            
    duration = time.perf_counter() - start
    
    # Actually print the text so you can see it's doing work!
    print(f"\n[{name} Output Preview]:\n{full_text}...\n")
    
    return [name, round(ttft, 2), round(tokens/duration, 2), round(duration, 2)]

def benchmark_native_pytorch(warm_up=3):
    print("\n>>> Testing Native PyTorch (AutoAWQ)...")
    from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer
    from threading import Thread
    import torch
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH, 
        device_map="cuda:0",
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True
    )
    
    # 1. Apply the Chat Template to force Hindi instruction following
    messages = [{"role": "user", "content": BENCHMARK_PROMPT}]
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")
    
    # Warm-up
    for i in range(warm_up):
        print(f"  Warm-up {i+1}/3...", end="\r")
        model.generate(**tokenizer("hi", return_tensors="pt").to("cuda"), max_new_tokens=1)
    
    print("\n>>> Running real benchmark...")
    
    # 2. Use a Streamer to catch the exact TTFT
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True)
    generation_kwargs = dict(inputs, streamer=streamer, max_new_tokens=150, do_sample=False)
    
    # Start generation in a background thread
    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    
    start = time.perf_counter()
    thread.start()
    
    ttft = None
    tokens = 0
    full_text = ""
    
    # Read the stream as it's generated
    for text in streamer:
        if ttft is None and text.strip():
            ttft = (time.perf_counter() - start) * 1000
        full_text += text
        tokens += len(text.split())
        
    duration = time.perf_counter() - start
    
    print(f"\n[Native PyTorch Output Preview]:\n{full_text}...\n")
    
    del model
    torch.cuda.empty_cache()
    
    return ["Native PyTorch", round(ttft, 2), round(tokens/duration, 2), round(duration, 2)]# --- EXECUTION ---
results = []

try:
    results.append(benchmark_api("Ollama", OLLAMA_URL, "mashriram/sarvam-1"))
except Exception as e:
    print(f"Ollama Server not found: {e}")

try:
    results.append(benchmark_native_pytorch())
except Exception as e:
    print(f"Native PyTorch failed: {e}")

print("\n" + "="*60)
print("FINAL PERFORMANCE COMPARISON: RTX 3050 (4GB VRAM)")
print("="*60)
headers = ["Runtime", "Warm TTFT (ms)", "Tokens/Sec", "Total Time (s)"]
print(tabulate(results, headers=headers, tablefmt="grid"))

C:\Users\Ansh\AppData\Roaming\Python\Python312\site-packages\awq\__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)



>>> Testing Ollama...

[Ollama Output Preview]:

 Backpropagation: "कंप्यूटिंग मशीनरी और इंटेलिजेंस पर घोषणापत्र" के रूप में भी जाना जाने वाला बैकपीड्राॅसिग, एक तंत्रिका जाल को प्रशिक्षित करने की प्रक्रिया में उपयोग किया जाने वाला एक एल्गोरिथ्म है। इसे यह सुनिश्चित करने के लिए डिज़ाइन किया गया था कि मॉडल ने अपने वजन और पूर्वाभिज्याओं को समायोजित करके अपनी गलतियों से सीखा हो - अनिवार्य रूप से यह देखने के माध्यम से कि नेटवर्क नए डेटा पर कैसे प्रदर्शन करता है बनाम उसके आउटपुट के आधार पर त्रुटि कार्यों का मूल्यांकन करता है, भविष्यवाणियों में सुधार करना लक्ष्य।

हिन्दी में इसका क्या मतलब हैः "बैकप्रोग्रैस" शब्द को आसान भाषा में सरल बनाता है: हम यहाँ नेटवर्क के कनेक्शन के वजन और पूर्वाभ्यासों (जो हमने कुछ मापदंडों पर आधारित सीखा है) को समायोजित करने की चर्चा कर रहे हैं, यह सुनिश्चित करते हुए कि वे नेटवर्क का उपयोग करने वाले व्यक्ति ने आउटपुट-विशिष्ट गलतियों को सुधारने के लिए किया था।

एल्गोरिदम महत्वपूर्ण क्यों हैः "बैकप्रोग्रैशन" महत्वपूर्ण है क्योंकि: बड़े भाषा मॉडलों (एल.एल.एम.) में, जो 

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


C:\Users\Ansh\AppData\Roaming\Python\Python312\site-packages\transformers\generation\configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
C:\Users\Ansh\AppData\Roaming\Python\Python312\site-packages\transformers\generation\configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.



>>> Running real benchmark...

[Native PyTorch Output Preview]:
बैकप्रोपगेशन एक महत्वपूर्ण अवधारणा है जो डीप लर्निंग में उपयोग की जाती है, विशेष रूप से जब बड़े भाषा मॉडल को प्रशिक्षित किया जाता है। इसे सरल शब्दों में समझाने के लिए, कल्पना करें कि आप एक जटिल मशीन लर्निंग समस्या को हल करने की कोशिश कर रहे हैं, जैसे कि एक विशाल भूलभुलैया में नेविगेट करना।

बैकप्रोग्रैशन एक एल्गोरिदम है जो आपको यह समझने में मदद करता है कि आपके मॉडल के भीतर प्रत्येक परत कैसे काम करती है और वे एक-दूसरे के साथ कैसे परस्पर क्रिया करते हैं। यह भूलभुलैया में एक मार्गदर्शक की तरह है, जो आपको दिखाता है कि आप कहाँ गए हैं और आपको बताता है कि आप कहाँ जा...


FINAL PERFORMANCE COMPARISON: RTX 3050 (4GB VRAM)
+----------------+------------------+--------------+------------------+
| Runtime        |   Warm TTFT (ms) |   Tokens/Sec |   Total Time (s) |
+================+==================+==============+==================+
| Ollama         |           256.57 |        52.61 |             6.58 |
+----------------+--------

In [2]:
import time
import torch
import sys
import types
import functools
import requests
from tabulate import tabulate
import transformers.utils
import transformers.activations

# --- MONKEY PATCHES ---
fake_qwen3 = types.ModuleType("transformers.models.qwen3")
sys.modules["transformers.models.qwen3"] = fake_qwen3
sys.modules["transformers.models.qwen3.modeling_qwen3"] = fake_qwen3
setattr(fake_qwen3, "Qwen3DecoderLayer", object)
setattr(fake_qwen3, "Qwen3ForCausalLM", object)

transformers.utils.cached_property = functools.cached_property
if not hasattr(transformers.activations, 'PytorchGELUTanh'):
    transformers.activations.PytorchGELUTanh = transformers.activations.GELUActivation

from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer, TextIteratorStreamer
from threading import Thread

# --- CONFIGURATION ---
MODEL_PATH = "C:/Users/Ansh/Models/sarvam-1"
OLLAMA_URL = "http://localhost:11434/api/generate"
BENCHMARK_PROMPT = """
Explain the concept of 'Backpropagation' in Deep Learning using a simple analogy in Hindi. 
Then, list 3 reasons why it is essential for training Large Language Models.
"""

def benchmark_ollama(warm_up=3):
    print("\n>>> Testing Ollama (mashriram/sarvam-1)...")
    
    # Warm-up
    for i in range(warm_up):
        print(f"  Warm-up {i+1}/3...", end="\r")
        requests.post(OLLAMA_URL, json={"model": "mashriram/sarvam-1", "prompt": "hi", "stream": False})
    
    print("\n>>> Running real benchmark...")
    start_time = time.perf_counter()
    
    response = requests.post(OLLAMA_URL, json={
        "model": "mashriram/sarvam-1", 
        "prompt": BENCHMARK_PROMPT, 
        "stream": False,
        "options": {"temperature": 0}
    }).json()
    
    total_time = time.perf_counter() - start_time
    
    # Extract native Ollama metrics
    ttft = (response.get("prompt_eval_duration", 0) / 1e6) # Convert nanoseconds to ms
    tps = response.get("eval_count", 0) / (response.get("eval_duration", 1) / 1e9)
    tpot = (response.get("eval_duration", 0) / 1e6) / max(1, response.get("eval_count", 1)) # ms per token
    prefill_speed = response.get("prompt_eval_count", 0) / (response.get("prompt_eval_duration", 1) / 1e9)
    
    # Ollama uses GGUF mmap, meaning the OS manages VRAM dynamically. 
    # For a 2B Q4 model + Context, it typically hovers around 1.8GB.
    peak_vram = "~1.8 GB (Dynamic)" 
    
    return ["Ollama (GGUF)", round(ttft, 2), round(tps, 2), round(tpot, 2), round(prefill_speed, 2), peak_vram]

def benchmark_native_pytorch(warm_up=3):
    print("\n>>> Testing Native PyTorch (AutoAWQ)...")
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    from transformers import AutoModelForCausalLM
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH, 
        device_map="cuda:0",
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True
    )
    
    messages = [{"role": "user", "content": BENCHMARK_PROMPT}]
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")
    prompt_tokens = len(inputs.input_ids[0])
    
    # Warm-up
    for i in range(warm_up):
        print(f"  Warm-up {i+1}/3...", end="\r")
        model.generate(**tokenizer("hi", return_tensors="pt").to("cuda"), max_new_tokens=1)
    
    print("\n>>> Running real benchmark...")
    torch.cuda.reset_peak_memory_stats() # Start tracking VRAM
    
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True)
    thread = Thread(target=model.generate, kwargs=dict(inputs, streamer=streamer, max_new_tokens=100, do_sample=False))
    
    start = time.perf_counter()
    thread.start()
    
    ttft = None
    tokens = 0
    for text in streamer:
        if ttft is None and text.strip():
            ttft = (time.perf_counter() - start) * 1000
        tokens += len(text.split())
        
    duration = time.perf_counter() - start
    
    # PyTorch Metrics
    tps = tokens / duration
    tpot = (duration * 1000) / max(1, tokens)
    prefill_speed = prompt_tokens / (ttft / 1000) if ttft else 0
    peak_vram = f"{torch.cuda.max_memory_allocated() / (1024**3):.2f} GB"
    
    del model
    torch.cuda.empty_cache()
    
    return ["Native PyTorch", round(ttft, 2), round(tps, 2), round(tpot, 2), round(prefill_speed, 2), peak_vram]

# --- EXECUTION ---
results = [benchmark_ollama(), benchmark_native_pytorch()]

print("\n" + "="*85)
print("FINAL PERFORMANCE COMPARISON: RTX 3050 (4GB VRAM)")
print("="*85)
headers = ["Runtime", "TTFT (ms)", "Tokens/Sec", "TPOT (ms/token)", "Prefill (Tokens/s)", "Peak VRAM"]
print(tabulate(results, headers=headers, tablefmt="grid"))


>>> Testing Ollama (mashriram/sarvam-1)...
  Warm-up 3/3...
>>> Running real benchmark...

>>> Testing Native PyTorch (AutoAWQ)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.



>>> Running real benchmark...

FINAL PERFORMANCE COMPARISON: RTX 3050 (4GB VRAM)
+----------------+-------------+--------------+-------------------+----------------------+-------------------+
| Runtime        |   TTFT (ms) |   Tokens/Sec |   TPOT (ms/token) |   Prefill (Tokens/s) | Peak VRAM         |
+================+=============+==============+===================+======================+===================+
| Ollama (GGUF)  |       30.16 |        58.02 |             17.23 |              1624.9  | ~1.8 GB (Dynamic) |
+----------------+-------------+--------------+-------------------+----------------------+-------------------+
| Native PyTorch |     4301.81 |         1.15 |            866.43 |                11.39 | 4.82 GB           |
+----------------+-------------+--------------+-------------------+----------------------+-------------------+
